# Hugging Face PEFT LoRA Supervised Fine-Tuning

This notebook turns the LoRA concept into a real supervised fine-tuning workflow with Hugging Face Transformers and PEFT. The default model is intentionally tiny so the notebook can be read and dry-run on modest machines. For real work, replace it with an instruction model that matches your task and hardware.

## What You Will Learn

By the end, you should understand:

- how PEFT attaches LoRA adapters,
- how to choose target modules,
- how to format SFT examples,
- how to configure a small Trainer run,
- how to save adapter weights,
- and what to check before scaling to a larger model.

## 1. Install Notes

This notebook uses optional fine-tuning packages. Install them when you are ready to run the training cells:

```bash
pip install transformers datasets accelerate peft
```

The notebook catches missing packages so you can still study the flow without installing everything immediately.

In [ ]:
import importlib.util
from pathlib import Path

import pandas as pd

def has_package(name):
    return importlib.util.find_spec(name) is not None

required = ["torch", "transformers", "datasets", "accelerate", "peft"]
pd.DataFrame({"package": required, "installed": [has_package(name) for name in required]})

## 2. Choose a Model

For learning, use a tiny model. For real SFT, choose a chat/instruction model whose license and hardware requirements fit your project.

Examples:

| Model | Use | Notes |
|---|---|---|
| `sshleifer/tiny-gpt2` | Notebook mechanics | Tiny, not a real chat model |
| `HuggingFaceTB/SmolLM2-135M-Instruct` | Small instruction demo | Better for chat-style examples |
| `Qwen/Qwen2.5-0.5B-Instruct` | Practical small demo | More realistic, more memory |
| `mistralai/Mistral-7B-Instruct-v0.3` | Real fine-tuning target | GPU memory required |

In [ ]:
model_name = "sshleifer/tiny-gpt2"
output_dir = Path("finetuning_outputs/lora_sft")
output_dir.mkdir(parents=True, exist_ok=True)
model_name, output_dir

## 3. Build a Tiny SFT Dataset

Real fine-tuning should use carefully reviewed examples. Here we use a tiny synthetic dataset to make the code path clear. The important part is the schema and formatting, not the quality of this toy data.

In [ ]:
records = [
    {"instruction": "Explain LoRA in one paragraph.", "response": "LoRA freezes a pretrained model and trains small low-rank adapter matrices that modify selected layers during fine-tuning."},
    {"instruction": "What is QLoRA?", "response": "QLoRA fine-tunes LoRA adapters while loading the base model in 4-bit precision to reduce GPU memory usage."},
    {"instruction": "Why do we use eval sets during fine-tuning?", "response": "Evaluation sets reveal whether training improves held-out behavior rather than merely memorizing the training examples."},
    {"instruction": "What is catastrophic forgetting?", "response": "Catastrophic forgetting happens when fine-tuning damages useful behavior the base model previously had."},
    {"instruction": "When should I use RAG instead of fine-tuning?", "response": "Use RAG when the model needs access to changing external facts rather than a new behavior pattern."},
]
pd.DataFrame(records)

In [ ]:
def format_sft_text(record):
    return (
        "### Instruction:\n"
        f"{record['instruction']}\n\n"
        "### Response:\n"
        f"{record['response']}"
    )

formatted_records = [{"text": format_sft_text(record)} for record in records]
print(formatted_records[0]["text"])

## 4. Load Dataset, Tokenizer, and Model

The code below is guarded. If packages are missing, it prints the install guidance instead of failing the whole notebook.

In [ ]:
try:
    import torch
    from datasets import Dataset
    from transformers import AutoModelForCausalLM, AutoTokenizer

    dataset = Dataset.from_list(formatted_records).train_test_split(test_size=0.4, seed=42)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(model_name)
    model.config.pad_token_id = tokenizer.pad_token_id
    print(dataset)
    print(f"pad_token: {tokenizer.pad_token!r}")
except Exception as error:
    dataset = None
    tokenizer = None
    model = None
    print("Model setup skipped. Install transformers datasets accelerate peft, then rerun.")
    print(f"Reason: {error}")

## 5. Tokenize Examples

This simple setup trains on the whole formatted text. For chat SFT, you often want assistant-only loss masking. That is why the previous notebook matters. Many high-level trainers can handle this if configured correctly.

In [ ]:
def tokenize_batch(batch):
    tokenized = tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=128,
    )
    tokenized["labels"] = [ids.copy() for ids in tokenized["input_ids"]]
    return tokenized

try:
    tokenized_dataset = dataset.map(tokenize_batch, batched=True, remove_columns=["text"])
    print(tokenized_dataset)
except Exception as error:
    tokenized_dataset = None
    print("Tokenization skipped. Run the setup cell first.")
    print(f"Reason: {error}")

## 6. Attach LoRA with PEFT

PEFT handles adapter injection. For tiny GPT-2, target module names differ from Llama-style models. In real Llama/Qwen/Mistral models, you will often target modules such as `q_proj`, `v_proj`, `o_proj`, `gate_proj`, `up_proj`, and `down_proj`.

For `sshleifer/tiny-gpt2`, we target `c_attn` because GPT-2 combines query, key, and value projection in one module.

In [ ]:
try:
    from peft import LoraConfig, TaskType, get_peft_model

    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        target_modules=["c_attn"],
    )
    peft_model = get_peft_model(model, lora_config)
    peft_model.print_trainable_parameters()
except Exception as error:
    peft_model = None
    print("PEFT setup skipped. Run model setup and install peft first.")
    print(f"Reason: {error}")

## 7. Configure a Small Trainer Run

This is intentionally tiny. Real training needs more examples, larger batch planning, careful learning rate selection, eval metrics, and checkpoint management.

In [ ]:
try:
    from transformers import DataCollatorForLanguageModeling, Trainer, TrainingArguments

    training_args = TrainingArguments(
        output_dir=str(output_dir),
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=1,
        learning_rate=2e-4,
        num_train_epochs=1,
        logging_steps=1,
        eval_strategy="epoch",
        save_strategy="no",
        report_to=[],
    )

    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
    trainer = Trainer(
        model=peft_model,
        args=training_args,
        train_dataset=tokenized_dataset["train"],
        eval_dataset=tokenized_dataset["test"],
        data_collator=data_collator,
    )
    print("Trainer is ready. Uncomment trainer.train() when you want to run the tiny demo.")
    # trainer.train()
except Exception as error:
    trainer = None
    print("Trainer setup skipped. Complete previous setup cells first.")
    print(f"Reason: {error}")

## 8. Save Adapter Weights

PEFT normally saves only the adapter weights and adapter config. This is much smaller than saving the full model. You need the original base model plus the adapter to reproduce behavior.

In [ ]:
try:
    adapter_dir = output_dir / "adapter"
    peft_model.save_pretrained(adapter_dir)
    tokenizer.save_pretrained(adapter_dir)
    print(f"saved adapter to: {adapter_dir.resolve()}")
    print([path.name for path in adapter_dir.iterdir()])
except Exception as error:
    print("Adapter save skipped. Create a PEFT model first.")
    print(f"Reason: {error}")

## 9. Scaling Checklist

Before replacing the tiny model with a real model, check these:

- Does the tokenizer use the right chat template?
- Are labels masked correctly?
- Are target module names correct for the model family?
- Does the dataset have enough high-quality examples?
- Is eval data held out and realistic?
- Is the learning rate small enough for adapter tuning?
- Is gradient accumulation set to reach the desired effective batch size?
- Are checkpoints and adapter versions named clearly?

## Summary

PEFT gives you the practical LoRA workflow: load base model, format data, tokenize examples, attach adapters, train only adapter weights, and save the adapter. The tiny demo is for mechanics; the same structure scales to real instruction models when your data, hardware, and evaluation are ready.

Next, we will use QLoRA to reduce memory by loading the base model in 4-bit precision while training LoRA adapters.